### https://docs.ray.io/en/latest/cluster/vms/user-guides/community/spark.html


Config:
- 2 Workers: 110 GB Memory, 16 Cores
- 1 Driver: 220 GB Memory, 32 Cores
Runtime:
- 16.4 LTS ML
Type:
- Standard_NC16as_T4_v3 [T4]


Notes: 

- We recommend setting the argument num_cpus_worker_node to the number of CPU cores per Apache Spark worker node. Similarly, setting num_gpus_worker_node to the number of GPUs per Apache Spark worker node is optimal. With this configuration, each Apache Spark worker node launches one Ray worker node that will fully utilize the resources of each Apache Spark worker node.
- Set the environment variable RAY_memory_monitor_refresh_ms to 0 within the Databricks cluster configuration when starting your Apache Spark cluster.


- In each spark worker node, we recommend making the sum of 'spark_executor_memory + num_Ray_worker_nodes_per_spark_worker * (memory_worker_node + object_store_memory_worker_node)' to be less than 'spark_worker_physical_memory * 0.8', otherwise it might lead to spark worker physical memory exhaustion and Ray task OOM errors.

In [0]:
# You configured 'spark.task.resource.gpu.amount' to 1.0,we recommend setting this value to 0 so that Spark jobs do not reserve GPU resources, preventing Ray-on-Spark workloads from having the maximum number of GPUs available. 

spark.conf.set("spark.task.resource.gpu.amount", "0")

In [0]:
import ray

total_cores = int(spark.sparkContext.defaultParallelism)
num_workers = int(ray.util.spark.MAX_NUM_WORKER_NODES)
total_gpus = int(spark.sparkContext.getConf().get("spark.driver.resource.gpu.amount"))

print(f"Total cores: {total_cores}")
print(f"Total GPUs: {total_gpus}")
print(f"Total workers: {num_workers}")

In [0]:
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

num_workers = 1

setup_ray_cluster(
  # Set min and max to disable autoscaling
  min_worker_nodes=num_workers, 
  max_worker_nodes=num_workers,
  num_cpus_per_node=total_cores // num_workers,
  num_gpus_per_node=total_gpus,
  num_cpus_head_node=total_cores // num_workers,
  num_gpus_head_node=total_gpus,
  collect_log_to_path="/dbfs/tmp/ray_collected_logs"
)

# Pass any custom Ray configuration with ray.init
ray.init(ignore_reinit_error=True)

In [0]:
%sh nvidia-smi

In [0]:
import ray
import ray.data
from typing import Dict
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import os
import yaml

config_path = "../local_config.yaml"
with open(config_path, 'r') as config_file:
    config = yaml.safe_load(config_file)

os.environ['HF_DATASETS_CACHE'] = config.get("cifar_cache")
dataset_path = "uoft-cs/cifar10"

In [0]:
# Fallback: Use helper functions and convert to Ray Data
from utils import hf_dataset_utilities as hf_util

# Load with helper functions
cifar_dataset = hf_util.hfds_download_volume(
    hf_cache=os.environ['HF_DATASETS_CACHE'],
    dataset_path='uoft-cs/cifar10',
    trust_remote_code=True,
    disable_progress=False,
)

# Convert to Ray Data
ray_cifar_train = ray.data.from_huggingface(cifar_dataset["train"])
ray_cifar_test = ray.data.from_huggingface(cifar_dataset["test"])

print("✓ Dataset loaded with helper functions and converted to Ray Data")
print(f"Train dataset schema: {ray_cifar_train.schema()}")
print(f"Test dataset schema: {ray_cifar_test.schema()}")

# Inspect the data structure to understand the format
print("\n=== Inspecting Ray Data Structure ===")
sample_data = ray_cifar_train.take(1)
print(f"Sample data keys: {sample_data[0].keys()}")
print(f"Sample data types: {[(k, type(v)) for k, v in sample_data[0].items()]}")

if "img" in sample_data[0]:
    img_sample = sample_data[0]["img"]
    print(f"Image type: {type(img_sample)}")
    if hasattr(img_sample, 'shape'):
        print(f"Image shape: {img_sample.shape}")
    elif hasattr(img_sample, 'size'):
        print(f"Image size: {img_sample.size}")
    elif isinstance(img_sample, dict):
        print(f"Image dict keys: {img_sample.keys()}")

print(f"Label: {sample_data[0]['label']}, type: {type(sample_data[0]['label'])}")

#### Note:  Ray Data's from_huggingface() creates a different batch structure than expected. Instead of iterating over individual images, the batch contains arrays/lists where batch["img"] is a list of images, not a single image.

In [0]:
def transform_cifar_batch(batch: Dict[str, np.ndarray]) -> Dict[str, torch.Tensor]:
    """Transform a batch of CIFAR images using torchvision transforms."""
    
    # Define the transform pipeline using the helper logic
    transform_list = [
        transforms.Resize((32, 32)),                # Resize to specified size
        transforms.RandomHorizontalFlip(),          # Random horizontal flip
        transforms.ToTensor()                       # Convert PIL image to Tensor
    ]
    # Convert grayscale to RGB if needed
    transform_list.append(
        transforms.Lambda(lambda x: x.repeat(3, 1, 1) if x.size(0) == 1 else x)
    )
    
    # Normalize using ImageNet stats
    transform_list.append(
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    )
    transform_pipeline = transforms.Compose(transform_list)
    
    # Transform images in the batch
    transformed_images = []
    
    # Handle the batch format from Ray Data
    # Ray Data from HuggingFace creates batches where each column is a list/array
    images_data = batch["img"]
    
    # Check if images_data is a single item or a batch
    if not isinstance(images_data, (list, np.ndarray)):
        images_data = [images_data]
    
    for img in images_data:
        try:
            # Handle different image formats
            if isinstance(img, dict):
                # If img is a dict, it might have nested structure like {"bytes": ..., "path": ...}
                if "bytes" in img:
                    # Handle bytes format
                    from io import BytesIO
                    img = Image.open(BytesIO(img["bytes"]))
                elif "path" in img:
                    # Handle path format
                    img = Image.open(img["path"])
                else:
                    # Try to get the first value if it's an unknown dict format
                    img_value = list(img.values())[0]
                    if isinstance(img_value, np.ndarray):
                        img = Image.fromarray(img_value)
                    else:
                        raise ValueError(f"Unknown dict image format: {img.keys()}")
            elif isinstance(img, np.ndarray):
                # Convert numpy array to PIL Image
                img = Image.fromarray(img)
            elif hasattr(img, 'convert'):
                # Already a PIL Image
                pass
            else:
                raise ValueError(f"Unknown image type: {type(img)}")
            
            # Ensure image is in RGB mode
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Apply transforms
            transformed_img = transform_pipeline(img)
            transformed_images.append(transformed_img)
            
        except Exception as e:
            print(f"Error processing image of type {type(img)}: {e}")
            # Create a dummy tensor as fallback
            dummy_tensor = torch.zeros(3, 32, 32)
            transformed_images.append(dummy_tensor)
    
    # Stack tensors and return
    batch["image"] = torch.stack(transformed_images)
    
    # Handle labels - ensure they're in the right format
    labels_data = batch["label"]
    if not isinstance(labels_data, (list, np.ndarray)):
        labels_data = [labels_data]

    labels_array = np.array(labels_data, dtype=np.int64)
    batch["label"] = torch.from_numpy(labels_array)
        
    # Remove original img column to save memory
    if "img" in batch:
        del batch["img"]
    
    return batch

In [0]:
print("\nApplying transformations using Ray Data...")

# First, test the transformation function on a small sample
print("Testing transformation function on a small sample...")
try:
    test_batch = ray_cifar_train.take_batch(batch_size=2)
    print(f"Test batch keys: {test_batch.keys()}")
    print(f"Test batch img type: {type(test_batch['img'])}")
    print(f"Test batch img length: {len(test_batch['img'])}")
    print(f"First image type: {type(test_batch['img'][0])}")
    
    # Test the transformation function
    transformed_test = transform_cifar_batch(test_batch)
    print("✓ Transformation function test passed!")
    print(f"Transformed keys: {transformed_test.keys()}")
    print(f"Image tensor shape: {transformed_test['image'].shape}")
    print(f"Label tensor shape: {transformed_test['label'].shape}")
    
except Exception as e:
    print(f"Transformation function test failed: {e}")
    import traceback
    traceback.print_exc()

# If test passed, apply to full datasets
print("\nApplying transformations to full datasets...")
try:
    # Transform the datasets with smaller batch size for safety
    ray_train_transformed = ray_cifar_train.map_batches(
        transform_cifar_batch,
        batch_size=50,   # Smaller batch size to avoid memory issues
        num_cpus=1       # Use 1 CPU to avoid conflicts
    )

    ray_test_transformed = ray_cifar_test.map_batches(
        transform_cifar_batch,
        batch_size=50,
        num_cpus=1
    )

    print("✓ Transformations applied successfully")
    print(f"Transformed train schema: {ray_train_transformed.schema()}")
    print(f"Transformed test schema: {ray_test_transformed.schema()}")
    
    # Test a sample from the transformed dataset
    print("\nTesting transformed dataset...")
    sample_transformed = ray_train_transformed.take(1)
    print(f"Sample transformed keys: {sample_transformed[0].keys()}")
    print(f"Sample image shape: {sample_transformed[0]['image'].shape}")
    print(f"Sample label: {sample_transformed[0]['label']}")
    
except Exception as e:
    print(f"Full transformation failed: {e}")
    import traceback
    traceback.print_exc()


In [0]:
ray_train_transformed.show(limit=1)

In [0]:
ray_test_transformed.show(limit=1)

In [0]:
print("\nTesting transformed data...")
sample_batch = ray_train_transformed.take(2)
print(f"Sample batch size: {len(sample_batch)}")
print(f"Sample keys: {sample_batch[0].keys()}")
print(f"Image tensor shape: {sample_batch[0]['image'].shape}")
print(f"Label tensor type: {type(sample_batch[0]['label'])}")
print(f"Label value: {sample_batch[0]['label']}")

In [0]:
# Time Ray Data approach
from utils.hf_dataset_utilities import Timer

print("Timing Ray Data batch iteration...")
ray_timer = Timer()
batch_count = 0
for batch in ray_train_transformed.iter_torch_batches(batch_size=64, local_shuffle_buffer_size=1000):
    batch_count += 1
    if batch_count >= 10:  # Test first 10 batches
        break
ray_time = ray_timer.stop()
print(f"Ray Data: {ray_time:.4f} seconds for {batch_count} batches")

In [0]:
print("Timing traditional PyTorch DataLoader...")
from torch.utils.data import DataLoader

# Create traditional PyTorch dataset for comparison
CIFARDataset = hf_util.create_torch_image_dataset(
    image_key="img",
    label_key="label"
)

ds_transforms = hf_util.default_image_transforms(
    image_size=32,
    normalize_transform=True,
    convert_rgb=True
)

if 'cifar_dataset' in locals():
    torch_train_dataset = CIFARDataset(cifar_dataset['train'], transform=ds_transforms)
    torch_dataloader = DataLoader(torch_train_dataset, batch_size=64, shuffle=True, num_workers=2)
    
    torch_timer = Timer()
    batch_count = 0
    for batch in torch_dataloader:
        batch_count += 1
        if batch_count >= 10:  # Test first 10 batches
            break
    torch_time = torch_timer.stop()
    print(f"PyTorch DataLoader: {torch_time:.4f} seconds for {batch_count} batches")
    
    # Performance comparison
    speedup = torch_time / ray_time if ray_time > 0 else float('inf')
    print(f"Ray Data is {speedup:.2f}x {'faster' if speedup > 1 else 'slower'} than PyTorch DataLoader")


# Training

In [0]:
# Training function using Ray Data shards for distributed training
import os
import tempfile
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torchvision.models import resnet18
import ray.train.torch
import ray.train

def train_func_ray_data():
    """Training function that uses Ray Data shards for distributed training."""
    
    # Model setup
    model = resnet18(num_classes=10)
    model = ray.train.torch.prepare_model(model)
    criterion = CrossEntropyLoss()
    
    # Optimizer setup
    learning_rate = 1e-4
    optimizer = Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    
    # Get the dataset shards for this worker
    train_data_shard = ray.train.get_dataset_shard("train")
    test_data_shard = ray.train.get_dataset_shard("test")
    
    print(f"Worker {ray.train.get_context().get_world_rank()}: Got dataset shards")
    print(f"Train shard schema: {train_data_shard.schema()}")
    
    # Training loop
    num_epochs = 3  # Reduced epochs for faster testing
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        
        # Iterate over batches using Ray Data shard
        # The data is already transformed with images and labels separated
        for batch in train_data_shard.iter_torch_batches(
            batch_size=64,
            dtypes={"image": torch.float32, "label": torch.long},  # Specify dtypes for each column
            local_shuffle_buffer_size=1000,
            drop_last=False
        ):
            # Extract features and labels from the batch
            # Our transformed data has 'image' and 'label' columns
            images = batch["image"]  # Shape: [batch_size, 3, 32, 32]
            labels = batch["label"]  # Shape: [batch_size]
            
            # CRITICAL FIX: Ensure labels are Long tensors for CrossEntropyLoss
            if labels.dtype != torch.long:
                labels = labels.long()
            
            # Debug print for first batch
            if num_batches == 0:
                print(f"Worker {ray.train.get_context().get_world_rank()}: "
                      f"Images shape: {images.shape}, dtype: {images.dtype}")
                print(f"Worker {ray.train.get_context().get_world_rank()}: "
                      f"Labels shape: {labels.shape}, dtype: {labels.dtype}")
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            # Print progress occasionally
            if num_batches % 50 == 0:
                print(f"Worker {ray.train.get_context().get_world_rank()}: "
                      f"Epoch {epoch}, Batch {num_batches}, Loss: {loss.item():.4f}")
        
        # Calculate average loss
        avg_loss = total_loss / max(num_batches, 1)
        
        # Evaluation on test shard
        model.eval()
        correct = 0
        total = 0
        test_loss = 0.0
        test_batches = 0
        
        with torch.no_grad():
            for batch in test_data_shard.iter_torch_batches(
                batch_size=64, 
                dtypes={"image": torch.float32, "label": torch.long},  # Specify dtypes for each column
                drop_last=False
            ):
                images = batch["image"]
                labels = batch["label"]
                
                if labels.dtype != torch.long:
                    labels = labels.long()
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                test_batches += 1
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        accuracy = 100 * correct / total if total > 0 else 0
        avg_test_loss = test_loss / max(test_batches, 1)
        
        # Report metrics
        metrics = {
            "train_loss": avg_loss,
            "test_loss": avg_test_loss,
            "accuracy": accuracy,
            "epoch": epoch,
            "num_train_batches": num_batches,
            "num_test_batches": test_batches
        }

        # ray.train.report(metrics)

        # Save checkpoint - simplified approach to avoid DBFS issues
        try:
            with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
                model_path = os.path.join(temp_checkpoint_dir, "model.pt")
                
                # Save model state dict
                if isinstance(model, torch.nn.DataParallel) or isinstance(model, torch.nn.parallel.DistributedDataParallel):
                    torch.save(model.module.state_dict(), model_path)
                else:
                    torch.save(model.state_dict(), model_path)
                
                # Create checkpoint
                checkpoint = ray.train.Checkpoint.from_directory(temp_checkpoint_dir)
                
                # Report metrics with checkpoint
                ray.train.report(metrics, checkpoint=checkpoint)
                
        except Exception as checkpoint_error:
            print(f"Warning: Checkpoint save failed: {checkpoint_error}")
            # Report metrics without checkpoint as fallback
            ray.train.report(metrics)
        
        if ray.train.get_context().get_world_rank() == 0:
            print(f"Epoch {epoch}: Train Loss={avg_loss:.4f}, Test Loss={avg_test_loss:.4f}, Accuracy={accuracy:.2f}%")

print("✓ Ray Data training function defined!")


In [0]:
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

# TODO UPDATE VARIABLES IN SCALING CONFIG
# Configure scaling for distributed training
scaling_config = ScalingConfig(
    num_workers=2,
    use_gpu=True,   # Enable GPU training
    # resources_per_worker={"CPU": 16, "GPU": 1}  # Resources per worker
)

storage_dir = "/tmp/ray_ws_logs_ray_data_training"
try:
  os.makedirs(storage_dir, exist_ok=True)
  print(f"Storage directory created: {storage_dir}")
except Exception as e:
  print(f"Error creating storage directory: {e}")

# Configure run settings
run_config = RunConfig(
    storage_path=storage_dir, 
    name="cifar_resnet_ray_data",
    # Stop after 3 epochs or if accuracy reaches 70%
    stop={"epoch": 3, "accuracy": 70.0},
    checkpoint_config=ray.train.CheckpointConfig(
        num_to_keep=2,  # Keep only the 2 best checkpoints
        checkpoint_score_attribute="accuracy",
        checkpoint_score_order="max"
    )
)

# Ensure we have the transformed datasets
if 'ray_train_transformed' not in locals():
    print("Transformed datasets not found. Please run the transformation cells first.")
else:
    # Create datasets dictionary for Ray Train using the transformed Ray Data
    datasets = {
        "train": ray_train_transformed,
        "test": ray_test_transformed
    }
    
    print("✓ Ray Data datasets prepared:")
    print(f"  - Train dataset schema: {datasets['train'].schema()}")
    print(f"  - Test dataset schema: {datasets['test'].schema()}")
    print(f"  - Scaling config: {scaling_config}")
    print(f"  - Run config: {run_config}")
    
    # Create the Ray Train trainer
    trainer = TorchTrainer(
        train_func_ray_data,
        scaling_config=scaling_config,
        run_config=run_config,
        datasets=datasets  
    )
    
    print("✓ Ray Train trainer created successfully!")
    print("\nTrainer configuration:")
    print(f"  - Training function: {train_func_ray_data.__name__}")
    print(f"  - Number of workers: {scaling_config.num_workers}")
    print(f"  - GPU enabled: {scaling_config.use_gpu}")
    print(f"  - Dataset sharding: Automatic via Ray Data")


In [0]:
# Execute distributed training with Ray Data
print("=== Starting Distributed Training with Ray Data ===")

if 'trainer' not in locals():
    print("Trainer not found. Please run the trainer setup cell first.")
else:
    try:
        print("Launching distributed training...")
        
        result = trainer.fit()
        
        print("Training completed successfully!")
        print("\n=== Training Results ===")
        print(f"Final metrics: {result.metrics}")
        print(f"Best checkpoint: {result.checkpoint}")
        print(f"Training logs path: {result.path}")
        
        if result.error:
            print(f"Training error: {result.error}")
        else:
            print("No errors during training!")
            
    except Exception as e:
        print(f"Training failed with error: {e}")
        import traceback
        traceback.print_exc()


In [0]:
# Load and evaluate the trained model
print("=== Model Evaluation and Analysis ===")

if 'result' not in locals():
    print("Training result not found. Please run the training cell first.")
else:
    try:
        # Load the best trained model
        print("Loading the best trained model...")
        with result.checkpoint.as_directory() as checkpoint_dir:
            model_state_dict = torch.load(os.path.join(checkpoint_dir, "model.pt"))
            model = resnet18(num_classes=10)
            model.load_state_dict(model_state_dict)
            model.eval()
            print("✓ Model loaded successfully!")

        # If checkpointing is disabled: 
        # print("Creating a fresh model for evaluation (checkpointing was disabled)...")
        # model = resnet18(num_classes=10)
        # model.eval()
        # print("✓ Fresh model created for evaluation!")
        
        # Test the model on a few samples from Ray Data
        print("\n=== Testing Model on Ray Data Samples ===")
        test_samples = ray_test_transformed.take(5)
        
        class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                      'dog', 'frog', 'horse', 'ship', 'truck']
        
        with torch.no_grad():
            for i, sample in enumerate(test_samples):
                image = sample['image']
                if isinstance(image, np.ndarray):
                    image = torch.from_numpy(image)
                image = image.unsqueeze(0)  # Add batch dimension
                label = sample['label']
                
                output = model(image)
                _, predicted = torch.max(output, 1)
                confidence = torch.softmax(output, 1).max().item()
                
                true_class = class_names[label]
                pred_class = class_names[predicted.item()]
                
                print(f"Sample {i+1}:")
                print(f"  True: {true_class} (label: {label})")
                print(f"  Predicted: {pred_class} (label: {predicted.item()})")
                print(f"  Confidence: {confidence:.3f}")
                print(f"  Correct: {'✓' if label == predicted.item() else '✗'}")
                print()
        
        # Display final training metrics
        print(f"\n=== Final Training Metrics ===")
        final_metrics = result.metrics
        if final_metrics:
            for key, value in final_metrics.items():
                if isinstance(value, float):
                    print(f"{key}: {value:.4f}")
                else:
                    print(f"{key}: {value}")
        
    except Exception as e:
        print(f"Model evaluation failed: {e}")
        import traceback
        traceback.print_exc()

# Shutdown to free up resources

In [0]:
print("Shutting down Ray cluster...")

try:
    ray.util.spark.shutdown_ray_cluster()
    print("✓ Ray cluster shutdown complete!")
except Exception as e:
    print(f"Warning: Error during cleanup: {e}")